In [1]:
import sys
sys.path.append("/home/shaikhq/research/QueryFormer")

In [2]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import time
import pandas as pd
from scipy.stats import pearsonr
from model.util import Normalizer
from model.database_util import get_hist_file, get_job_table_sample, collator
from model.model import QueryFormer
from model.database_util import Encoding
from model.dataset import PlanTreeDataset
from model.trainer import eval_workload, train, train_single

In [3]:
!wget https://raw.githubusercontent.com/IBM/db2-jupyter/master/db2.ipynb
%run db2.ipynb
db2creds_file = 'db2con.env'
from dotenv import dotenv_values
db2creds = dotenv_values(db2creds_file)

--2024-06-19 09:13:51--  https://raw.githubusercontent.com/IBM/db2-jupyter/master/db2.ipynb


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 156491 (153K) [text/plain]
Saving to: ‘db2.ipynb.10’

db2.ipynb.10        100%[===================>] 152.82K  --.-KB/s    in 0.01s   

2024-06-19 09:13:51 (11.2 MB/s) - ‘db2.ipynb.10’ saved [156491/156491]



<>:1692: SyntaxWarning: invalid escape sequence '\s'
<>:2285: SyntaxWarning: invalid escape sequence '\?'
/tmp/ipykernel_159227/1557473136.py:1692: SyntaxWarning: invalid escape sequence '\s'
  firstCommand = "(?:^\s*)([a-zA-Z]+)(?:\s+.*|$)"
/tmp/ipykernel_159227/1557473136.py:2285: SyntaxWarning: invalid escape sequence '\?'
  pattern = "\?\*[0-9]+"


Db2 Extensions Loaded. Version: 2024-05-29


In [4]:
%sql CONNECT CREDENTIALS db2creds

Connection successful. tpcds @ localhost 


In [5]:
df_queries_columns = ['query_id', 'appl_id', 'uow_id', 'activity_id', 'explain_time', 'query']

In [6]:
df_queries = pd.read_csv('success.csv', header=None, names=df_queries_columns)

In [7]:
df_queries.shape

(5764, 6)

In [8]:
df_queries.head(5)

,query_id,appl_id,uow_id,activity_id,explain_time,query
0,1,*LOCAL.shaikhq.240530151621,4,1,2024-05-30-08.16.11.238384,"SELECT TPCDS.CUSTOMER.C_BIRTH_YEAR , TPCDS.DAT..."
1,2,*LOCAL.shaikhq.240530151621,12,1,2024-05-30-08.16.12.871381,"SELECT TPCDS.CATALOG_PAGE.CP_END_DATE_SK , TPC..."
2,4,*LOCAL.shaikhq.240530151621,24,1,2024-05-30-08.16.24.574183,"SELECT TPCDS.WEB_SITE.WEB_CLOSE_DATE_SK , TPCD..."
3,5,*LOCAL.shaikhq.240530151621,32,1,2024-05-30-08.16.25.771665,"SELECT TPCDS.CATALOG_PAGE.CP_END_DATE_SK , TPC..."
4,6,*LOCAL.shaikhq.240530151621,40,1,2024-05-30-08.16.26.997895,"SELECT TPCDS.CATALOG_PAGE.CP_CATALOG_PAGE_SK ,..."


In [9]:
query1_ts = "2024-05-30-08.16.25.771665"
df_queries = df_queries[df_queries['explain_time'] == query1_ts]
query_id = df_queries['query_id'].values[0]
activity_id = df_queries['activity_id'].values[0]
appl_id = df_queries['appl_id'].values[0]
uow_id = df_queries['uow_id'].values[0]

In [10]:
query_id

5

# collecting query level stats

In [11]:
# collecting final actual card
sql = f""" 
SELECT ROWS_RETURNED, 
    SORT_SHRHEAP_TOP 
FROM ACTIVITY_DB2ACTIVITIES 
WHERE ACTIVITY_ID = {activity_id} AND 
APPL_ID = '{appl_id}' AND 
UOW_ID = {uow_id}
"""

In [12]:
print(sql)

 
SELECT ROWS_RETURNED, 
    SORT_SHRHEAP_TOP 
FROM ACTIVITY_DB2ACTIVITIES 
WHERE ACTIVITY_ID = 1 AND 
APPL_ID = '*LOCAL.shaikhq.240530151621' AND 
UOW_ID = 32



In [13]:

df_activity = %sql {sql}

In [14]:

actual_card = df_activity['ROWS_RETURNED'].values[0]
sort_shrheap_top = df_activity['SORT_SHRHEAP_TOP'].values[0]

In [15]:
sql = f""" 
SELECT STMT_EXEC_TIME 
FROM ACTIVITYMETRICS_DB2ACTIVITIES 
WHERE ACTIVITY_ID = {activity_id} AND 
APPL_ID = '{appl_id}' AND 
UOW_ID = {uow_id}
"""

df_activitymetrics = %sql {sql}
stmt_exec_time = df_activitymetrics['STMT_EXEC_TIME'].values[0]

In [16]:
print('actual_card: {}'.format(actual_card))
print('sort_shrheap_top: {}'.format(sort_shrheap_top))
print('stmt_exec_time: {}'.format(stmt_exec_time))

actual_card: 108
sort_shrheap_top: 0
stmt_exec_time: 2


# Collecting Node level information for each node

## Get the list of operator_ids for the current query

In [17]:
# find out the the nodes / operators
# fetching operators
sql = f"""
SELECT OPERATOR_ID, OPERATOR_TYPE 
FROM EXPLAIN_OPERATOR
WHERE EXPLAIN_TIME = '{query1_ts}'
"""

df_explain_operator = %sql {sql}
print(df_explain_operator)
operator_ids = df_explain_operator['OPERATOR_ID'].tolist()
print(operator_ids)

   OPERATOR_ID OPERATOR_TYPE
0            1        RETURN
1            2        NLJOIN
2            3        FETCH 
3            4        IXSCAN
4            5        TBSCAN
[1, 2, 3, 4, 5]


In [18]:
df_explain_stream = pd.read_csv('EXPLAIN_STREAM.csv')
stream_cols = ['SOURCE_ID', 'TARGET_TYPE', 'TARGET_ID', 'OBJECT_NAME', 'STREAM_COUNT', 'COLUMN_COUNT', 'COLUMN_NAMES']
df_explain_stream = df_explain_stream[(df_explain_stream['EXPLAIN_TIME'] == query1_ts)][stream_cols]

In [19]:
# df_explain_stream.head()

In [20]:
# df_explain_predicate = pd.read_csv('EXPLAIN_PREDICATE.csv')

In [21]:
# df_explain_predicate['HOW_APPLIED'].unique()

In [22]:
df_explain_predicate = pd.read_csv('EXPLAIN_PREDICATE.csv')
predicate_cols = [ 'OPERATOR_ID',
       'PREDICATE_ID', 'HOW_APPLIED', 'WHEN_EVALUATED', 'RELOP_TYPE',
       'SUBQUERY', 'FILTER_FACTOR', 'PREDICATE_TEXT']

df_predicate_filtered = df_explain_predicate[df_explain_predicate['EXPLAIN_TIME'] == query1_ts][predicate_cols]
print('df_predicate_filtered shape :{}'.format(df_predicate_filtered.shape))
print('df_predicate_filtered: ', df_predicate_filtered)

df_predicate_filtered shape :(5, 8)
df_predicate_filtered:      OPERATOR_ID  PREDICATE_ID HOW_APPLIED WHEN_EVALUATED RELOP_TYPE SUBQUERY  \
13            2             2  JOIN                              EQ        N   
14            3             5  SARG                              LE        N   
15            4             3  START                             EQ        N   
16            4             3  STOP                              EQ        N   
17            5             2  SARG                              EQ        N   

    FILTER_FACTOR                      PREDICATE_TEXT  
13       1.000000  (Q1.CP_END_DATE_SK = Q2.D_DATE_SK)  
14       0.888848          (Q2.D_LAST_DOM <= 2481252)  
15       0.000014            (Q2.D_DATE_SK = 2451449)  
16       0.000014            (Q2.D_DATE_SK = 2451449)  
17       1.000000  (Q1.CP_END_DATE_SK = Q2.D_DATE_SK)  


In [23]:
# df_predicate_filtered

In [24]:
# op_id = 3
# df_predicate_filtered[df_predicate_filtered['OPERATOR_ID'] == op_id]['PREDICATE_TEXT'].values

In [25]:
# df_explain_stream.shape

In [26]:
# print(df_explain_stream['COLUMN_NAMES'][0])

In [27]:
nodes = {}

for operator_id in operator_ids:
    node_dict = {}
    node_dict['Node Type'] = df_explain_operator[df_explain_operator['OPERATOR_ID'] == operator_id]['OPERATOR_TYPE'].values[0]
    
    # check if there is any relation / table involved in this operation
    relation_name = df_explain_stream[(df_explain_stream['TARGET_ID'] == operator_id) 
                                      & (df_explain_stream['OBJECT_NAME'].notna())]['OBJECT_NAME'].values
    
    if len(relation_name) > 0:
        node_dict['Relation Name'] = relation_name[0]
        # collecting local predicates, if any
        local_predicate = df_predicate_filtered[df_predicate_filtered['OPERATOR_ID'] == operator_id]['PREDICATE_TEXT'].values
        if len(local_predicate) > 0:
            node_dict['Filter'] = ' AND '.join(local_predicate.tolist())
        
    # check if there is any join predicate
    join_predicate = df_predicate_filtered[(df_predicate_filtered['OPERATOR_ID'] == operator_id) & 
                          (df_predicate_filtered['HOW_APPLIED'].str.strip() == 'JOIN')]['PREDICATE_TEXT'].values
    
    if len(join_predicate) > 0:
        node_dict['Join Predicate'] = join_predicate[0]
    
    nodes[operator_id] = node_dict


In [28]:
nodes

{1: {'Node Type': 'RETURN'},
 2: {'Node Type': 'NLJOIN',
  'Join Predicate': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)'},
 3: {'Node Type': 'FETCH ',
  'Relation Name': 'DATE_DIM2',
  'Filter': '(Q2.D_LAST_DOM <= 2481252)'},
 4: {'Node Type': 'IXSCAN',
  'Relation Name': 'SQL240509072221830',
  'Filter': '(Q2.D_DATE_SK = 2451449) AND (Q2.D_DATE_SK = 2451449)'},
 5: {'Node Type': 'TBSCAN',
  'Relation Name': 'CATALOG_PAGE',
  'Filter': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)'}}

In [29]:
df_explain_stream

,SOURCE_ID,TARGET_TYPE,TARGET_ID,OBJECT_NAME,STREAM_COUNT,COLUMN_COUNT,COLUMN_NAMES
23,-1,O,4,SQL240509072221830,73049.000000,2,+Q2.$RID$+Q2.D_DATE_SK
24,4,O,3,NaN,1.000000,-1,NaN
25,-1,O,3,DATE_DIM2,73049.000000,2,+Q2.D_DOW+Q2.D_LAST_DOM
26,3,O,2,NaN,1.000000,-1,NaN
27,-1,O,5,CATALOG_PAGE,11718.000000,2,+Q1.$RID$+Q1.CP_END_DATE_SK
28,5,O,2,NaN,111.320999,-1,NaN
29,2,O,1,NaN,111.320999,-1,NaN


In [30]:

# Identifying parent-child relationship
stream_cols = ['SOURCE_ID', 'TARGET_ID']
df_stream_filtered = df_explain_stream[stream_cols]

In [31]:
for key, value in nodes.items():
    print(f'{key}: {value}')

1: {'Node Type': 'RETURN'}
2: {'Node Type': 'NLJOIN', 'Join Predicate': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)'}
3: {'Node Type': 'FETCH ', 'Relation Name': 'DATE_DIM2', 'Filter': '(Q2.D_LAST_DOM <= 2481252)'}
4: {'Node Type': 'IXSCAN', 'Relation Name': 'SQL240509072221830', 'Filter': '(Q2.D_DATE_SK = 2451449) AND (Q2.D_DATE_SK = 2451449)'}
5: {'Node Type': 'TBSCAN', 'Relation Name': 'CATALOG_PAGE', 'Filter': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)'}


# Scratchpad section

In [32]:
df_explain_stream = pd.read_csv('EXPLAIN_STREAM.csv')

In [33]:
df_explain_stream.columns

Index(['EXPLAIN_REQUESTER', 'EXPLAIN_TIME', 'SOURCE_NAME', 'SOURCE_SCHEMA',
       'SOURCE_VERSION', 'EXPLAIN_LEVEL', 'STMTNO', 'SECTNO', 'STREAM_ID',
       'SOURCE_TYPE', 'SOURCE_ID', 'TARGET_TYPE', 'TARGET_ID', 'OBJECT_SCHEMA',
       'OBJECT_NAME', 'STREAM_COUNT', 'COLUMN_COUNT', 'PREDICATE_ID',
       'COLUMN_NAMES', 'PMID', 'SINGLE_NODE', 'PARTITION_COLUMNS',
       'SEQUENCE_SIZES', 'OBJECT_TENANTID'],
      dtype='object')

In [34]:
stream_cols = ['SOURCE_ID', 'TARGET_ID']
df_stream_filtered = df_explain_stream[(df_explain_stream['EXPLAIN_TIME'] == query1_ts)][stream_cols]

In [35]:
nodes

{1: {'Node Type': 'RETURN'},
 2: {'Node Type': 'NLJOIN',
  'Join Predicate': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)'},
 3: {'Node Type': 'FETCH ',
  'Relation Name': 'DATE_DIM2',
  'Filter': '(Q2.D_LAST_DOM <= 2481252)'},
 4: {'Node Type': 'IXSCAN',
  'Relation Name': 'SQL240509072221830',
  'Filter': '(Q2.D_DATE_SK = 2451449) AND (Q2.D_DATE_SK = 2451449)'},
 5: {'Node Type': 'TBSCAN',
  'Relation Name': 'CATALOG_PAGE',
  'Filter': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)'}}

In [36]:
for index, row in df_stream_filtered.iterrows():
    source_id = row['SOURCE_ID']
    target_id = row['TARGET_ID']
    if source_id > 0:
        if 'Plans' not in nodes[target_id]:
            nodes[target_id]['Plans'] = []
        nodes[target_id]['Plans'].append(source_id)
    #print('source: {}, target: {}'.format(source_id, target_id))

In [37]:
nodes[5]['Filter'] = '(Q1.CP_CATALOG_NUMBER = 5)'

In [38]:
for key, value in nodes.items():
    print(f'{key}: {value}')

1: {'Node Type': 'RETURN', 'Plans': [2]}
2: {'Node Type': 'NLJOIN', 'Join Predicate': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)', 'Plans': [3, 5]}
3: {'Node Type': 'FETCH ', 'Relation Name': 'DATE_DIM2', 'Filter': '(Q2.D_LAST_DOM <= 2481252)', 'Plans': [4]}
4: {'Node Type': 'IXSCAN', 'Relation Name': 'SQL240509072221830', 'Filter': '(Q2.D_DATE_SK = 2451449) AND (Q2.D_DATE_SK = 2451449)'}
5: {'Node Type': 'TBSCAN', 'Relation Name': 'CATALOG_PAGE', 'Filter': '(Q1.CP_CATALOG_NUMBER = 5)'}


In [39]:
nodes

{1: {'Node Type': 'RETURN', 'Plans': [2]},
 2: {'Node Type': 'NLJOIN',
  'Join Predicate': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)',
  'Plans': [3, 5]},
 3: {'Node Type': 'FETCH ',
  'Relation Name': 'DATE_DIM2',
  'Filter': '(Q2.D_LAST_DOM <= 2481252)',
  'Plans': [4]},
 4: {'Node Type': 'IXSCAN',
  'Relation Name': 'SQL240509072221830',
  'Filter': '(Q2.D_DATE_SK = 2451449) AND (Q2.D_DATE_SK = 2451449)'},
 5: {'Node Type': 'TBSCAN',
  'Relation Name': 'CATALOG_PAGE',
  'Filter': '(Q1.CP_CATALOG_NUMBER = 5)'}}

In [40]:
json_df = pd.DataFrame(columns=['id', 'json'])

In [41]:
nodes

{1: {'Node Type': 'RETURN', 'Plans': [2]},
 2: {'Node Type': 'NLJOIN',
  'Join Predicate': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)',
  'Plans': [3, 5]},
 3: {'Node Type': 'FETCH ',
  'Relation Name': 'DATE_DIM2',
  'Filter': '(Q2.D_LAST_DOM <= 2481252)',
  'Plans': [4]},
 4: {'Node Type': 'IXSCAN',
  'Relation Name': 'SQL240509072221830',
  'Filter': '(Q2.D_DATE_SK = 2451449) AND (Q2.D_DATE_SK = 2451449)'},
 5: {'Node Type': 'TBSCAN',
  'Relation Name': 'CATALOG_PAGE',
  'Filter': '(Q1.CP_CATALOG_NUMBER = 5)'}}

In [42]:
nodes

{1: {'Node Type': 'RETURN', 'Plans': [2]},
 2: {'Node Type': 'NLJOIN',
  'Join Predicate': '(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)',
  'Plans': [3, 5]},
 3: {'Node Type': 'FETCH ',
  'Relation Name': 'DATE_DIM2',
  'Filter': '(Q2.D_LAST_DOM <= 2481252)',
  'Plans': [4]},
 4: {'Node Type': 'IXSCAN',
  'Relation Name': 'SQL240509072221830',
  'Filter': '(Q2.D_DATE_SK = 2451449) AND (Q2.D_DATE_SK = 2451449)'},
 5: {'Node Type': 'TBSCAN',
  'Relation Name': 'CATALOG_PAGE',
  'Filter': '(Q1.CP_CATALOG_NUMBER = 5)'}}

In [43]:
nodes[1]['Actual Rows'] = actual_card

In [44]:
import json

def build_tree(nodes, node_key):
    node = nodes[node_key].copy()  # Get the node and make a copy of it
    for key, value in node.items():  # Iterate over each item in the node
        if isinstance(value, np.int64):  # If the item is of type int64
            node[key] = int(value)  # Convert it to int
    if 'Plans' in node:  # If the node has children
        node['Plans'] = [build_tree(nodes, child_key) for child_key in node['Plans']]  # Replace child keys with child nodes
    return node

tree = {"Plan": build_tree(nodes, 1)}
tree['Execution Time'] = float(stmt_exec_time)
tree['sort_shrheap_top'] = float(sort_shrheap_top)

json_object = json.dumps(tree)

print(json_object)

{"Plan": {"Node Type": "RETURN", "Plans": [{"Node Type": "NLJOIN", "Join Predicate": "(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)", "Plans": [{"Node Type": "FETCH ", "Relation Name": "DATE_DIM2", "Filter": "(Q2.D_LAST_DOM <= 2481252)", "Plans": [{"Node Type": "IXSCAN", "Relation Name": "SQL240509072221830", "Filter": "(Q2.D_DATE_SK = 2451449) AND (Q2.D_DATE_SK = 2451449)"}]}, {"Node Type": "TBSCAN", "Relation Name": "CATALOG_PAGE", "Filter": "(Q1.CP_CATALOG_NUMBER = 5)"}]}], "Actual Rows": 108}, "Execution Time": 2.0, "sort_shrheap_top": 0.0}


In [45]:
json_df.loc[len(json_df)] = [query_id, json_object]

In [46]:
json_df

,id,json
0,5,"{""Plan"": {""Node Type"": ""RETURN"", ""Plans"": [{""N..."


In [47]:
json_parsed = json.loads(json_df['json'].iloc[0])
json_pretty = json.dumps(json_parsed, indent=4)
print(json_pretty)

with open('output.json', 'w') as f:
    f.write(json_pretty)

{
    "Plan": {
        "Node Type": "RETURN",
        "Plans": [
            {
                "Node Type": "NLJOIN",
                "Join Predicate": "(Q1.CP_END_DATE_SK = Q2.D_DATE_SK)",
                "Plans": [
                    {
                        "Node Type": "FETCH ",
                        "Relation Name": "DATE_DIM2",
                        "Filter": "(Q2.D_LAST_DOM <= 2481252)",
                        "Plans": [
                            {
                                "Node Type": "IXSCAN",
                                "Relation Name": "SQL240509072221830",
                                "Filter": "(Q2.D_DATE_SK = 2451449) AND (Q2.D_DATE_SK = 2451449)"
                            }
                        ]
                    },
                    {
                        "Node Type": "TBSCAN",
                        "Relation Name": "CATALOG_PAGE",
                        "Filter": "(Q1.CP_CATALOG_NUMBER = 5)"
                    }
                

In [48]:
nodes[1]

{'Node Type': 'RETURN', 'Plans': [2], 'Actual Rows': 108}

# TESTING - dataset generation

In [49]:
# loading the imdb encoding file
encoding_ckpt = torch.load('../../checkpoints/encoding.pt')
encoding = encoding_ckpt['encoding']

In [50]:
encoding

In [51]:
#TODO: change this path later for TPCDS dataset
imdb_path = '../imdb/'

In [52]:
#TODO: generate hist file for TPCDS dataset
hist_file = get_hist_file(imdb_path + 'histogram_string.csv')

In [53]:
#TODO: generate table samples for TPCDS dataset
table_sample = get_job_table_sample(imdb_path+'train')

Loaded queries with len  100000
Loaded bitmaps


In [54]:
import logging
import sys

In [55]:
# TODO: add these 2 normalizers for the Db2 training dataset
cost_norm = Normalizer()
card_norm = Normalizer()
to_predict = 'card'

In [56]:
encoding.column_min_max_vals['Q2.D_LAST_DOM'] = [2415020, 2488372]
encoding.column_min_max_vals['Q2.D_DATE_SK'] = [2415022, 2488070]
encoding.column_min_max_vals['Q1.CP_CATALOG_NUMBER'] = [1, 109]

In [57]:
sys.path.append("/home/shaikhq/research/QueryFormer")
import importlib
from model import dataset
importlib.reload(dataset)
from model.dataset import PlanTreeDataset

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

train_ds_single_query = PlanTreeDataset(json_df, None, encoding, hist_file, card_norm, cost_norm, to_predict, table_sample)

INFO:root:Initializing PlanTreeDataset
INFO:root:self.length = len(json_df): 1
INFO:root:nodes.type: <class 'list'>
INFO:root:number of nodes: 1
INFO:root:type of the first element in the list nodes: <class 'dict'>
INFO:root:keys in the first dictionary in the nodes list: dict_keys(['Node Type', 'Plans', 'Actual Rows'])
INFO:root:self.cards: [108]
INFO:root:self.card_labels: tensor([108])
min log(label): 0.6936470556015963
max log(label): 0.6936470556015963
INFO:root:idxs: [5]
INFO:root:beginning js_node2dict(self, idx, node): returns a dictionary of 4 tensors per query plan
INFO:root:returns a collated_dict of 4 tensors: 'x', 'attn_bias', 'rel_pos', 'heights 
INFO:root:nodeType: RETURN
INFO:root:typeId: 13
INFO:root:formatFilter - filters: []
INFO:root:formatFilter - alias: None
INFO:root:encode_filters() printing alias: None
INFO:root:formatJoin - join: None
INFO:root:formatJoin - joinId: 0
INFO:root:printing node features
INFO:root:node.typeId: 13
INFO:root:node.join: 0
INFO:root:no

# Unpacking the training dataset

In [58]:
type(train_ds_single_query[0])

tuple

In [59]:
len(train_ds_single_query[0])

2

In [60]:
train_ds_single_query[0][0].keys()

dict_keys(['x', 'attn_bias', 'rel_pos', 'heights'])

# Check the filter encoding info

In [61]:
train_ds_single_query[0][0]['x'].shape

torch.Size([1, 30, 1015])

In [62]:
train_ds_single_query[0][0]['x']

tensor([[[13.,  0., 20.,  ...,  0.,  0.,  0.],
         [14.,  0., 20.,  ...,  0.,  0.,  0.],
         [15.,  0., 21.,  ...,  0.,  0.,  0.],
         ...,
         [ 1.,  1.,  1.,  ...,  1.,  1.,  1.],
         [ 1.,  1.,  1.,  ...,  1.,  1.,  1.],
         [ 1.,  1.,  1.,  ...,  1.,  1.,  1.]]])

In [63]:
len(train_ds_single_query[0][0]['x'][0])

30

In [64]:
len(train_ds_single_query[0][0]['x'][0][0])

1015

In [65]:
print(train_ds_single_query[0][0]['x'][0][0])

tensor([13.,  0., 20.,  ...,  0.,  0.,  0.])


In [66]:
# Reset the print options to default
torch.set_printoptions(profile="full")

In [67]:
train_ds_single_query[0][0]['x'][0][0][0:15]

tensor([13.,  0., 20.,  0.,  0.,  3.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,
         0.])

In [68]:
# content of the first node - node features
train_ds_single_query[0][0]['x'][0][0]

tensor([13.,  0., 20.,  0.,  0.,  3.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  

In [69]:
type(train_ds_single_query[0][1])

tuple

In [70]:
train_ds_single_query.labels

tensor([108])

In [71]:
type(train_ds_single_query.labels)

torch.Tensor

# Executing the Model pipeline using the training dataset of a single example query plan

In [72]:
class Args:
    # bs = 1024
    # SQ: smaller batch size
    bs = 1
    lr = 0.001
    # epochs = 200
    epochs = 1
    clip_size = 50
    embed_size = 64
    pred_hid = 128
    ffn_dim = 128
    head_size = 12
    n_layers = 8
    dropout = 0.1
    sch_decay = 0.6
    # device = 'cuda:0'
    device = 'cpu'
    newpath = './results/full/cost/'
    to_predict = 'card'
args = Args()

In [73]:
model = QueryFormer(emb_size = args.embed_size ,ffn_dim = args.ffn_dim, head_size = args.head_size, \
                 dropout = args.dropout, n_layers = args.n_layers, \
                 use_sample = False, use_hist = False, \
                 pred_hid = args.pred_hid
                )

In [74]:
crit = nn.MSELoss()

In [75]:
crit = nn.MSELoss()
model = train_single(model, train_ds_single_query, train_ds_single_query, crit, cost_norm, args)

INFO:root:QuerfyFormer forward
INFO:root:x shape: torch.Size([1, 30, 1015])
INFO:root:getFilter: opsId tensor([[3, 0, 0],
        [3, 0, 0],
        [4, 0, 0],
        [1, 0, 0],
        [1, 1, 0],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]])


IndexError: index out of range in self